### General Imports

In [1]:
%load_ext autoreload
%autoreload 2
import sys, os, torch, datetime, time
import numpy as np
sys.path.append("..")
import rfi_ml_bmxM
import matplotlib.pyplot as plt

In [2]:
print(f"Is CUDA supported by this system?{torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

Is CUDA supported by this system?True
CUDA version: 12.1


### Loading Data

In [3]:
def loadData(self, data_dir, test=False):
        #print('Data dir:', data_dir+'/**/*_'+self.freq+'.npy')
        #data_files = glob.glob(data_dir+'/**/*_'+self.freq+'.npy')
        print('Data dir:', data_dir+'/**/*_'+self.freq+'.npy')
        data_files = glob.glob(data_dir+'/**/*_'+self.freq+'.npy')

In [4]:
# Downloading from main drive (Windows)

train_dir='C:/MLdatasets/bmxdata/train_data' #BMX data should already be chunked into 1024 sets
eval_dir='C:/MLdatasets/bmxdata/eval_data'

# Downloading from external drive
#train_dir='D:/MLdatasets/bmxdata/train_data'
#eval_dir='C:/MLdatasets/bmxdata/eval_data'


freq = '0000' #setting frequency channel

In [5]:
Nepochs = 20
Np = 2**11 #2k
hidden_dim = 2**10
z_dim = 16
Nfft = Np // 2 + 1

start_time = time.time()

b = rfi_ml_bmxM.BMXLoader(Np=Np, freq=freq)
train_data = b.loadData(data_dir=train_dir, test=False)
eval_data = b.loadData(data_dir=eval_dir, test=False)
train_data, eval_data = b.normalizeData(train_data, eval_data)

Data dir: C:/MLdatasets/bmxdata/train_data/**/*_0000.npy
Data Loaded!
Size:  (118552, 2048)
Data dir: C:/MLdatasets/bmxdata/eval_data/**/*_0000.npy
Data Loaded!
Size:  (89936, 2048)
Train data mean:  2.214976970311273e-17
Eval data mean:  2.0194637833732208e-16
RMS normalization factor:  0.055328948869969016


In [6]:
train_data.shape

(118552, 2048)

In [7]:
torch.stack([torch.from_numpy(train_data)]).shape

torch.Size([1, 118552, 2048])

### Training Parameters

In [8]:
n = rfi_ml_bmxM.RFIDetect(Np, Nepochs = Nepochs, z_dim = z_dim, hidden_dim = hidden_dim)

In [9]:
n.train(train_data)
del train_data

[  0/20][  0/29638]	Loss: 0.0032352670
[  0/20][100/29638]	Loss: 0.0014094487
[  0/20][200/29638]	Loss: 0.0020450763
[  0/20][300/29638]	Loss: 0.0017396436
[  0/20][400/29638]	Loss: 0.0022817785
[  0/20][500/29638]	Loss: 0.0011805757
[  0/20][600/29638]	Loss: 0.0022667018
[  0/20][700/29638]	Loss: 0.0012050753
[  0/20][800/29638]	Loss: 0.0016205341
[  0/20][900/29638]	Loss: 0.0014485763
[  0/20][1000/29638]	Loss: 0.0019942736
[  0/20][1100/29638]	Loss: 0.0011975734
[  0/20][1200/29638]	Loss: 0.0009559547
[  0/20][1300/29638]	Loss: 0.0010065401
[  0/20][1400/29638]	Loss: 0.0021014218
[  0/20][1500/29638]	Loss: 0.0007849882
[  0/20][1600/29638]	Loss: 0.0006219725
[  0/20][1700/29638]	Loss: 0.0012666344
[  0/20][1800/29638]	Loss: 0.0012927530
[  0/20][1900/29638]	Loss: 0.0011882272
[  0/20][2000/29638]	Loss: 0.0014801244
[  0/20][2100/29638]	Loss: 0.0018014305
[  0/20][2200/29638]	Loss: 0.0015569179
[  0/20][2300/29638]	Loss: 0.0017993341
[  0/20][2400/29638]	Loss: 0.0016415280
[  0/20][2

[ 28/30][2288/3704]	Loss: 0.0008406959
[ 28/30][2388/3704]	Loss: 0.0010133712
[ 28/30][2488/3704]	Loss: 0.0008416968
[ 28/30][2588/3704]	Loss: 0.0010374152
[ 28/30][2688/3704]	Loss: 0.0008188323
[ 28/30][2788/3704]	Loss: 0.0009955462
[ 28/30][2888/3704]	Loss: 0.0006471197
[ 28/30][2988/3704]	Loss: 0.0006365239
[ 28/30][3088/3704]	Loss: 0.0008489238
[ 28/30][3188/3704]	Loss: 0.0009150091
[ 28/30][3288/3704]	Loss: 0.0010482401
[ 28/30][3388/3704]	Loss: 0.0007661519
[ 28/30][3488/3704]	Loss: 0.0005866800
[ 28/30][3588/3704]	Loss: 0.0010024542
[ 28/30][3688/3704]	Loss: 0.0009817907
[ 29/30][ 84/3704]	Loss: 0.0006679685
[ 29/30][184/3704]	Loss: 0.0008751776
[ 29/30][284/3704]	Loss: 0.0008707471
[ 29/30][384/3704]	Loss: 0.0008816547
[ 29/30][484/3704]	Loss: 0.0008936792
[ 29/30][584/3704]	Loss: 0.0010395136
[ 29/30][684/3704]	Loss: 0.0006760212
[ 29/30][784/3704]	Loss: 0.0009414683
[ 29/30][884/3704]	Loss: 0.0008330989
[ 29/30][984/3704]	Loss: 0.0007271161
[ 29/30][1084/3704]	Loss: 0.0006828

In [10]:
start_test = time.time()
rfi_recov = n.evaluate(eval_data)
end_test = time.time()

Epochs:  20
N tests:  89936
Length of timestream:  2048
Input Timestream RMS:  [0.01387627 0.01158426 0.01277735 ... 0.03350664 0.02007465 0.05432052]
Avg RMS for all tests:  0.033699054 



In [ ]:
n.plot_eval(rfi_recov, eval_data)

Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_0.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_1.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_2.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_3.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_4.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_5.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_6.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_7.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_8.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_9.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_10.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_11.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_12.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_13.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_14.png
Saving file...rfi_ml/2024-08-06_10-33-54_overplot_test_15.png
Saving file...rfi_

Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_922.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_923.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_924.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_925.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_926.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_927.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_928.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_929.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_930.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_931.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_932.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_933.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_934.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_935.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_936.png
Saving file...rfi_ml/2022-08-15_11-50-35_overplot_test_

In [ ]:
print("Runtime: ", time.time()-start_time)
print("Evaltime: ", end_test-start_test)